# Evaluation of uncertainty of models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.default_params import *

In [ ]:
import functools
import math
import string

import IPython
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from properscoring import crps_ensemble
import xarray as xr
from xhistogram.xarray import histogram

from mlde_utils import cp_model_rotated_pole
from mlde_analysis import plot_map, SUBREGIONS, BOX_LOCATIONS
from mlde_analysis.bootstrap import resample_examples
from mlde_analysis.data import prep_eval_data
from mlde_analysis.display import pretty_table
from mlde_analysis.uncertainty import plot_spread_error, plot_scatter, compute_rmss_rmse_bins, se_bins, serat
from mlde_analysis.utils import chained_groupby_map

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, CPM_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

### Table: correlations

* pred domain mean & ensemble mean vs target domain mean
* pred domain mean vs target domain mean
* pred ensemble mean vs target
* pred vs target

In [ ]:
mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }

corr_coeff_das = []

for var in eval_vars:
    ds = VAR_DAS[var].sel(model=list(mois.keys()))

    pred_da = ds[f"pred_{var}"]
    target_da = ds[f"target_{var}"]
    
    corr_coeff_das.append(
        xr.merge([
            xr.corr(pred_da.mean(dim=["grid_latitude", "grid_longitude", "sample_id"]), target_da.mean(dim=["grid_latitude", "grid_longitude"]), dim=["ensemble_member", "time"]).rename(f"{var} domain mean sample mean corr"),
            xr.corr(pred_da.mean(dim=["grid_latitude", "grid_longitude"]), target_da.mean(dim=["grid_latitude", "grid_longitude"]), dim=["sample_id", "ensemble_member", "time"]).rename(f"{var} domain mean corr"),
            xr.corr(pred_da.mean(dim=["sample_id"]), target_da, dim=["ensemble_member", "time", "grid_latitude", "grid_longitude"]).rename(f"{var} sample mean corr"),
            xr.corr(pred_da, target_da, dim=["ensemble_member", "time", "grid_longitude", "grid_latitude", "sample_id"]).rename(f"{var} corr"),
            xr.corr(pred_da.max(dim=["grid_latitude", "grid_longitude"]).mean(dim=["sample_id"]), target_da.max(dim=["grid_latitude", "grid_longitude"]), dim=["ensemble_member", "time"]).rename(f"{var} mean domain max corr"),
        ]).expand_dims({"var": [var]})
    )

_ = pretty_table(xr.concat(corr_coeff_das, dim="var"), round=2)

## Figure: skill

* Domain mean scatter: samples ensemble mean vs CPM

In [ ]:
for var in eval_vars:
    scatter_fig = plt.figure(layout='constrained', figsize=(5.5, 5.5*(2/3.0)))
    # scatter_fig, ss_fig = fig.subfigures(1, 2, width_ratios=[2,1.075])
    
    mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }
    domain_mean_ensemble_mean_pred_da = PRED_DAS[var].sel(model=list(mois.keys())).mean(dim=["grid_latitude", "grid_longitude", "sample_id"], keep_attrs=True).assign_attrs({"long_name": "ML "+PRED_DAS[var].attrs['long_name']})
    domain_mean_target_da = CPM_DAS[var].mean(dim=["grid_latitude", "grid_longitude"], keep_attrs=True).assign_attrs({"long_name": "CPM "+CPM_DAS[var].attrs['long_name']})
    
    axd = scatter_fig.subplot_mosaic([mois.keys()], sharey=True, sharex=True)
    
    for idx, (model, model_emmean_pred_da) in enumerate(domain_mean_ensemble_mean_pred_da.groupby("model")):
        ax=axd[model]
        plot_scatter(
            pred_da=model_emmean_pred_da,
            target_da=domain_mean_target_da,
            ax=ax,
            line_props=mois[model],
        )
        ax.xaxis.set_tick_params(labelbottom=True)
        if idx > 0:
            ax.yaxis.label.set_visible(False)
        ax.annotate(
            f"{string.ascii_lowercase[idx]}.",
            xy=(-0.05, 1.04),
            xycoords=("axes fraction", "axes fraction"),
            weight="bold",
            ha="left",
            va="bottom",
        )
        
    mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) and not mconfig["deterministic"] }
    ds = VAR_DAS[var].sel(model=list(mois.keys()))
    pred_da = ds[f"pred_{var}"]
    target_da = ds[f"target_{var}"]

    bins = se_bins(pred_da, target_da, nbins=100)

    spread_error_ds = pred_da.groupby("model").map(compute_rmss_rmse_bins, target_da=target_da, bins=bins)

### Figure: domain mean scatter (pred vs CPM)
* Domain mean precip scatter: all pred samples vs CPM

In [ ]:
mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }

for var in eval_vars:
    scatter_fig = plt.figure(layout='constrained', figsize=(5.5, 5.5*(2/3.0)))

    pred_da = PRED_DAS[var].sel(model=list(mois.keys()))
    mean_pred_da = pred_da.mean(dim=["grid_latitude", "grid_longitude"], keep_attrs=True).assign_attrs({"long_name": "ML "+PRED_DAS[var].attrs['long_name']})
    target_mean_da = CPM_DAS[var].mean(dim=["grid_latitude", "grid_longitude"], keep_attrs=True).assign_attrs({"long_name": "CPM "+CPM_DAS[var].attrs['long_name']})
    
    axd = scatter_fig.subplot_mosaic([mois.keys()], sharey=True, sharex=True)
    
    for model, model_pred_da in mean_pred_da.groupby("model"):
        ax=axd[model]
        ax_idx = list(mois.keys()).index(model)
        plot_scatter(
            pred_da=model_pred_da,
            target_da=target_mean_da,
            ax=ax,
            line_props=mois[model],
        )
        ax.xaxis.set_tick_params(labelbottom=True)
        if ax_idx > 0:
            ax.yaxis.label.set_visible(False)
        
        ax.annotate(
            f"{string.ascii_lowercase[ax_idx]}.",
            xy=(-0.05, 1.04),
            xycoords=("axes fraction", "axes fraction"),
            weight="bold",
            ha="left",
            va="bottom",
        )

    plt.show()

### Domain max scatter
* Mean domain max precip scatter: max over domain mean over samples pred vs max over domain CPM

In [ ]:
mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }

for var in eval_vars:
    scatter_fig = plt.figure(layout='constrained', figsize=(5.5, 5.5*(2/3.0)))

    pred_da = PRED_DAS[var].sel(model=list(mois.keys()))
    pred_max_da = pred_da.max(dim=["grid_latitude", "grid_longitude"], keep_attrs=True).mean(dim=["sample_id"], keep_attrs=True).assign_attrs({"long_name": "ML "+PRED_DAS[var].attrs['long_name']})
    target_max_da = CPM_DAS[var].max(dim=["grid_latitude", "grid_longitude"], keep_attrs=True).assign_attrs({"long_name": "CPM "+CPM_DAS[var].attrs['long_name']})
    
    axd = scatter_fig.subplot_mosaic([mois.keys()], sharey=True, sharex=True)
    
    for idx, (model, model_pred_da) in enumerate(pred_max_da.groupby("model")):
        ax=axd[model]
        plot_scatter(
            pred_da=model_pred_da,
            target_da=target_max_da,
            ax=ax,
            line_props=mois[model],
        )
        ax.xaxis.set_tick_params(labelbottom=True)
        if idx > 0:
            ax.yaxis.label.set_visible(False)
        ax.annotate(
            f"{string.ascii_lowercase[idx]}.",
            xy=(-0.05, 1.04),
            xycoords=("axes fraction", "axes fraction"),
            weight="bold",
            ha="left",
            va="bottom",
        )

    plt.show()

### Figure: gridbox scatter (pred ensemble mean vs CPM)
* Grid box scatter: pred ensemble mean vs CPM

In [ ]:
for var in eval_vars:
    scatter_fig = plt.figure(layout='constrained', figsize=(5.5, 5.5*(2/3.0)))

    mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }
    emmean_pred_da = PRED_DAS[var].sel(model=list(mois.keys())).mean("sample_id", keep_attrs=True).assign_attrs({"long_name": "ML "+PRED_DAS[var].attrs['long_name']})
    target_da = CPM_DAS[var].assign_attrs({"long_name": "CPM "+CPM_DAS[var].attrs['long_name']})

    axd = scatter_fig.subplot_mosaic([mois.keys()], sharey=True, sharex=True)
    
    for idx, (model, model_pred_da) in enumerate(emmean_pred_da.groupby("model")):
        ax=axd[model]
        plot_scatter(
            pred_da=model_pred_da,
            target_da=target_da,
            ax=ax,
            line_props=mois[model],
            alpha=0.1,
        )
        ax.xaxis.set_tick_params(labelbottom=True)
        if idx > 0:
            ax.yaxis.label.set_visible(False)
        ax.annotate(
            f"{string.ascii_lowercase[idx]}.",
            xy=(-0.05, 1.04),
            xycoords=("axes fraction", "axes fraction"),
            weight="bold",
            ha="left",
            va="bottom",
        )

    plt.show()

### Figure: gridbox scatter (pred vs CPM)
* Grid box scatter: all pred samples vs CPM

In [ ]:
for var in eval_vars:
    scatter_fig = plt.figure(layout='constrained', figsize=(5.5, 5.5*(2/3.0)))

    mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }
    pred_da = PRED_DAS[var].sel(model=list(mois.keys())).assign_attrs({"long_name": "ML "+PRED_DAS[var].attrs['long_name']})
    target_da = CPM_DAS[var].assign_attrs({"long_name": "CPM "+CPM_DAS[var].attrs['long_name']})

    axd = scatter_fig.subplot_mosaic([mois.keys()], sharey=True, sharex=True)
    
    for idx, (model, model_pred_da) in enumerate(pred_da.groupby("model")):
        ax=axd[model]
        plot_scatter(
            pred_da=model_pred_da,
            target_da=target_da,
            ax=ax,
            line_props=mois[model],
            alpha=0.05,
        )
        ax.xaxis.set_tick_params(labelbottom=True)
        if idx > 0:
            ax.yaxis.label.set_visible(False)
        ax.annotate(
            f"{string.ascii_lowercase[idx]}.",
            xy=(-0.05, 1.04),
            xycoords=("axes fraction", "axes fraction"),
            weight="bold",
            ha="left",
            va="bottom",
        )

    plt.show()

## CRPS

In [ ]:
def group_crps(model_forecast_da, truth_da):
    return xr.apply_ufunc(
        crps_ensemble,
        truth_da,
        model_forecast_da.squeeze("model"),
        input_core_dims=[truth_da.dims, model_forecast_da.squeeze("model").dims],  # list with one entry per arg
        output_core_dims=[["examples", "grid_latitude", "grid_longitude"]],
        # vectorize=True,
    ).rename("CRPS").mean()

for var in eval_vars:
    print(var)
    
    mois = { model: mconfig for mconfigs in MODELS.values() for model, mconfig in mconfigs.items() if mconfig.get("UQ", True) }

    forecasts_da = PRED_DAS[var].sel(model=list(mois.keys())).stack(example=["ensemble_member", "time"]).transpose("model", "example", "grid_latitude", "grid_longitude", "sample_id") 
    crps_scores = {}

    truth = CPM_DAS[var].stack(example=["ensemble_member", "time"]).transpose("example", "grid_latitude", "grid_longitude")
    
    crps_scores = forecasts_da.groupby("model", squeeze=False).map(group_crps, truth_da=truth)
    pretty_table(crps_scores, round=4)

## Individual locations

In [ ]:
fig = plt.figure(layout="constrained", figsize=(1.5, 1.5))
ax = fig.subplots(subplot_kw={"projection": cp_model_rotated_pole})
ax.coastlines(**{"resolution": "10m", "linewidth": 0.3})
da = CPM_DAS[eval_vars[0]]
ax.set_extent((
    da.cf["X"].min(),
    da.cf["X"].max(),
    da.cf["Y"].min(),
    da.cf["Y"].max(),
))

for label, q in BOX_LOCATIONS.items():
    single_box_da = da.cf.sel(**q, method="nearest")
    ax.plot(single_box_da.cf["X"].values, single_box_da.cf["Y"].values, color='blue', markersize=0.5, marker='o', transform=cp_model_rotated_pole)
    ax.annotate(
        xy=(single_box_da.cf["X"].item(), single_box_da.cf["Y"].item()), xycoords="data",
        text=label, xytext=(0, -15), textcoords="offset pixels",
        ha='center',va="center", transform=cp_model_rotated_pole, fontsize="xx-small")
    
plt.show()

### Error distribution for individual locations

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"#### {var}", raw=True)

    fig = plt.figure(layout="constrained", figsize=(4.5, 3))
    axd = fig.subplot_mosaic([ [ f"{m} {l}" for m in MODELS["CPM"].keys() ] for l in BOX_LOCATIONS.keys() ], sharey=True, sharex=True)


        
    for i, (label, q) in enumerate(BOX_LOCATIONS.items()):
        ds = EVAL_DS["CPM"].cf.sel(**q, method="nearest")

        errors = ds[f"pred_{var}"] - ds[f"target_{var}"]
        max_err_mag = np.abs(errors).max().item()
        bins = np.arange(-math.ceil(max_err_mag), math.ceil(max_err_mag), 1)
        
        for model, model_errors_da in errors.groupby("model"):
            ax=axd[f"{model} {label}"]
            # ax.hist(model_errors_da.values.flat, alpha=0.5, bins=np.linspace(-250, 250, 501), density=True, label="model", histtype='step')
            model_errors_da.plot.hist(ax=ax, bins=bins, density=True, label=model, histtype='step', linewidth=0.5, color=MODELLABEL2SPEC[model]["color"])
            ax.set_title(model)
            ax.set_yscale("log")
    plt.show()

### Scatter for individual locations

In [ ]:
for var in eval_vars:
    seasonal_location_correlations = []
    IPython.display.display_markdown(f"#### {var}", raw=True)

    for label, q in BOX_LOCATIONS.items():
        ds = EVAL_DS["CPM"].cf.sel(**q, method="nearest")

        for season, season_ds in ds.groupby("time.season"):
            if season not in ["DJF", "JJA"]: continue

            IPython.display.display_markdown(f"##### {label} {season}", raw=True)
            fig = plt.figure(layout="constrained", figsize=(5.5, 1.5))
            axd = fig.subplot_mosaic(ds["model"].values.reshape(1,-1), sharey=True)
            
            seasonal_location_correlations.append(
                xr.corr(season_ds[f"pred_{var}"].mean(dim="sample_id"), season_ds[f"target_{var}"], dim=["ensemble_member", "time"]).rename(f"{var} domain mean corr").expand_dims({"location": [f"{label} {season}"]}),
            )
            
            for model, season_model_ds in season_ds.groupby("model"):
                pred_da = season_model_ds[f"pred_{var}"]
                cpm_da = season_model_ds[f"target_{var}"]
                if MODELLABEL2SPEC[model]["deterministic"]:
                    # don't let deterministic models appear brighter due to repeated predictions
                    pred_da = pred_da.isel(sample_id=[0])
                else:
                    # for generative models, consider the mean of the predictions
                    pred_da = pred_da.mean(dim="sample_id")
                
                ax = axd[model]
                ax.plot(
                    cpm_da.broadcast_like(pred_da).values.flat,
                    pred_da.values.flat,
                    alpha=0.1,
                    color=MODELLABEL2SPEC[model]["color"],
                    marker=".",
                    markersize=3,
                    linewidth=0,
                )
                ax.set_title(model, fontsize="small",)
                
                lims = [
                    np.min([ax.get_xlim(), ax.get_ylim()]),
                    np.max([ax.get_xlim(), ax.get_ylim()]),
                ]
                ax.set_aspect("equal")
                ax.set_xlim(lims)
                ax.set_ylim(lims)
                ax.plot(
                    [0, 1],
                    [0, 1],
                    transform=ax.transAxes,
                    linewidth=1,
                    color="black",
                    linestyle="--",
                    label="Ideal",
                )
    
            ax = list(axd.values())[0]
            ax.set_xlabel(
                f"CPM\n{xr.plot.utils.label_from_attrs(da=cpm_da)}",
                fontsize="xx-small",
            )
            ax.set_ylabel(
                f"ML mean prediction\n{xr.plot.utils.label_from_attrs(da=pred_da)}",
                fontsize="xx-small",
            )
            plt.show()
            
    pretty_table(xr.concat(seasonal_location_correlations, dim="location"), round=3,)